In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 3, Finished, Available, Finished, False)

# Read Bronze

In [5]:
from notebookutils import mssparkutils

# ============================================================
# FUNCTION: GET LATEST BRONZE FILE
# ============================================================
# Dynamically finds the latest date folder for a Bronze table
# and returns the complete Parquet file path.
#
# Example:
# Files/bronze/telemetry/2026-08-20/raw_telemetry.parquet
# ============================================================

def get_latest_bronze_path(table_name):

    base_path = f"Files/bronze/{table_name}"

    folders = mssparkutils.fs.ls(base_path)

    date_folders = [
        f.name.rstrip("/")
        for f in folders
        if f.isDir
    ]

    if not date_folders:
        raise Exception(
            f"No Bronze date folders found for {table_name}"
        )

    latest_date = max(date_folders)

    bronze_path = (
        f"{base_path}/"
        f"{latest_date}/"
        f"raw_{table_name}.parquet"
    )

    return bronze_path, latest_date

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 7, Finished, Available, Finished, False)

In [6]:
# ============================================================
# READ LATEST BRONZE TELEMETRY
# ============================================================

telemetry_path, telemetry_load_date = (
    get_latest_bronze_path("telemetry")
)

print("Bronze path:", telemetry_path)
print("Load date:", telemetry_load_date)

telemetry_df = spark.read.parquet(
    telemetry_path
)

display(telemetry_df.limit(5))

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 8, Finished, Available, Finished, False)

Bronze path: Files/bronze/telemetry/2026-08-20/raw_telemetry.parquet
Load date: 2026-08-20


SynapseWidget(Synapse.DataFrame, e1d59cc7-81e1-4272-8e8d-002066bb6cb1)

In [7]:
telemetry_df.createOrReplaceTempView("bronze_telemetry_latest")

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 9, Finished, Available, Finished, False)

In [8]:
# ============================================================
# READ LATEST BRONZE events
# ============================================================
events_path, events_load_date = (
    get_latest_bronze_path("events")
)

print("Bronze path:", events_path)
print("Load date:", events_load_date)

events_df = spark.read.parquet(
    events_path
)

display(events_df.limit(5))

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 10, Finished, Available, Finished, False)

Bronze path: Files/bronze/events/2026-08-20/raw_events.parquet
Load date: 2026-08-20


SynapseWidget(Synapse.DataFrame, 2b1a20ee-dc93-40e2-8b4e-bc0fd829f8be)

In [9]:
events_df.createOrReplaceTempView("bronze_events_latest")

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 11, Finished, Available, Finished, False)

In [10]:
# ============================================================
# READ LATEST BRONZE assest_metadata
# ============================================================
asset_metadata_path, asset_metadata_load_date = (
    get_latest_bronze_path("asset_metadata")
)
asset_metadata_df = spark.read.parquet(asset_metadata_path)

display(asset_metadata_df.limit(5))

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8caa4d64-9bde-44ec-af10-f2d904575c5b)

In [11]:
asset_metadata_df.createOrReplaceTempView(
    "bronze_asset_metadata_latest"
)

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 13, Finished, Available, Finished, False)

In [25]:
telemetry_df.printSchema()
events_df.printSchema()
asset_metadata_df.printSchema()

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 27, Finished, Available, Finished, False)

root
 |-- timestamp: string (nullable = true)
 |-- site_id: string (nullable = true)
 |-- building_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- humidity: string (nullable = true)
 |-- pressure: string (nullable = true)
 |-- vibration: string (nullable = true)
 |-- power_consumption: string (nullable = true)
 |-- operating_mode: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: string (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- message: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: string (nullable = true)

root
 |-- asset_id: string (nullable = true)
 |-- asset_name: string (nullable = true)
 |-- asset_type: str

# Data transformation

In [12]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.silver_telemetry AS

        SELECT
            raw_timestamp,
            timestamp,
            site_id,
            building_id,
            asset_id,
            sensor_id,
            temperature,
            humidity,
            pressure,
            vibration,
            power_consumption,
            operating_mode,
            source_file,
            ingestion_timestamp

        FROM
        (
            SELECT
                timestamp AS raw_timestamp,

                COALESCE(
                    to_timestamp(TRIM(timestamp), 'dd-MM-yyyy HH:mm'),
                    to_timestamp(TRIM(timestamp), 'dd-MM-yyyy HH:mm:ss'),
                    to_timestamp(TRIM(timestamp), 'yyyy-MM-dd HH:mm:ss'),
                    to_timestamp(TRIM(timestamp), 'yyyy-MM-dd HH:mm:ss.SSS')
                ) AS timestamp,

                NULLIF(TRIM(site_id), '') AS site_id,
                NULLIF(TRIM(building_id), '') AS building_id,
                NULLIF(TRIM(asset_id), '') AS asset_id,
                NULLIF(TRIM(sensor_id), '') AS sensor_id,

                CAST(temperature AS DOUBLE) AS temperature,
                CAST(humidity AS DOUBLE) AS humidity,
                CAST(pressure AS DOUBLE) AS pressure,
                CAST(vibration AS DOUBLE) AS vibration,

                COALESCE(
                    CAST(power_consumption AS DOUBLE),
                    0.0
                ) AS power_consumption,

                COALESCE(
                    NULLIF(UPPER(TRIM(operating_mode)), ''),
                    'UNKNOWN'
                ) AS operating_mode,

                'telemetry.csv' AS source_file,

                current_timestamp() AS ingestion_timestamp,

                ROW_NUMBER() OVER (
                    PARTITION BY
                        COALESCE(
                            to_timestamp(TRIM(timestamp), 'dd-MM-yyyy HH:mm'),
                            to_timestamp(TRIM(timestamp), 'dd-MM-yyyy HH:mm:ss'),
                            to_timestamp(TRIM(timestamp), 'yyyy-MM-dd HH:mm:ss'),
                            to_timestamp(TRIM(timestamp), 'yyyy-MM-dd HH:mm:ss.SSS')
                        ),
                        NULLIF(TRIM(site_id), ''),
                        NULLIF(TRIM(building_id), ''),
                        NULLIF(TRIM(asset_id), ''),
                        NULLIF(TRIM(sensor_id), '')
                    ORDER BY timestamp
                ) AS rn

            FROM bronze_telemetry_latest
        ) cleaned

        WHERE rn = 1
    """)
)

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0d5c13d7-afd0-49f8-bf59-3038fdee1d8b)

In [13]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.silver_events AS

        SELECT
            event_id,
            timestamp,
            asset_id,
            event_type,
            severity,
            message,
            source_file,
            ingestion_timestamp

        FROM
        (
            SELECT

                -- =====================================================
                -- 1. EVENT ID
                -- =====================================================
                -- Remove leading/trailing spaces.
                -- Convert empty strings to NULL.
                --
                -- We intentionally keep NULL values so that the
                -- DQ layer can identify missing event IDs.
                -- =====================================================

                NULLIF(TRIM(event_id), '') AS event_id,


                -- =====================================================
                -- 2. EVENT TIMESTAMP
                -- =====================================================
                -- Support multiple incoming timestamp formats.
                --
                -- If the timestamp cannot be converted using any
                -- supported format, the result becomes NULL.
                --
                -- DQ layer will identify these records as
                -- INVALID_TIMESTAMP.
                -- =====================================================

                COALESCE(

                    to_timestamp(
                        TRIM(`timestamp`),
                        'dd-MM-yyyy HH:mm'
                    ),

                    to_timestamp(
                        TRIM(`timestamp`),
                        'dd-MM-yyyy HH:mm:ss'
                    ),

                    to_timestamp(
                        TRIM(`timestamp`),
                        'yyyy-MM-dd HH:mm:ss'
                    ),

                    to_timestamp(
                        TRIM(`timestamp`),
                        'yyyy-MM-dd HH:mm:ss.SSS'
                    )

                ) AS timestamp,


                -- =====================================================
                -- 3. ASSET ID
                -- =====================================================
                -- Remove spaces.
                -- Convert empty values to NULL.
                --
                -- DQ layer will validate whether the asset ID exists
                -- and whether it is a valid asset.
                -- =====================================================

                NULLIF(TRIM(asset_id), '') AS asset_id,


                -- =====================================================
                -- 4. EVENT TYPE
                -- =====================================================
                -- Standardize event type to uppercase.
                -- Empty values become NULL.
                -- =====================================================

                UPPER(
                    NULLIF(TRIM(event_type), '')
                ) AS event_type,


                -- =====================================================
                -- 5. SEVERITY
                -- =====================================================
                -- Standardize severity to uppercase.
                -- Empty values become NULL.
                -- =====================================================

                UPPER(
                    NULLIF(TRIM(severity), '')
                ) AS severity,


                -- =====================================================
                -- 6. MESSAGE
                -- =====================================================
                -- If the business requirement allows a default value,
                -- replace missing/empty messages with 'NO MESSAGE'.
                -- =====================================================

                COALESCE(
                    NULLIF(TRIM(message), ''),
                    'NO MESSAGE'
                ) AS message,


                -- =====================================================
                -- 7. SOURCE FILE
                -- =====================================================
                -- Keep source information for data lineage/auditing.
                -- =====================================================

                'events.csv' AS source_file,


                -- =====================================================
                -- 8. INGESTION TIMESTAMP
                -- =====================================================
                -- Records when the data entered the Silver layer.
                -- Used for late-arriving-data analysis.
                -- =====================================================

                current_timestamp() AS ingestion_timestamp


            FROM bronze_events_latest
        ) cleaned
    """)
)

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d1126a54-b4ef-4466-9756-add133b96eb8)

In [14]:
display( 
    spark.sql(""" 
        CREATE OR REPLACE TABLE dbo.silver_asset_metadata AS 
 
        SELECT 
            NULLIF(TRIM(asset_id), '') AS asset_id, 
 
            NULLIF(TRIM(asset_name), '') AS asset_name, 
 
            COALESCE( 
                NULLIF(UPPER(TRIM(asset_type)), ''), 
                'UNKNOWN' 
            ) AS asset_type, 
 
            COALESCE( 
                NULLIF(TRIM(manufacturer), ''), 
                'UNKNOWN' 
            ) AS manufacturer, 
 
            COALESCE( 
                to_date(TRIM(installation_date), 'dd-MM-yyyy'), 
                to_date(TRIM(installation_date), 'yyyy-MM-dd'), 
                to_date(TRIM(installation_date), 'dd/MM/yyyy') 
            ) AS installation_date, 
 
            NULLIF(TRIM(site_id), '') AS site_id, 
 
            'asset_metadata.csv' AS source_file, 
 
            current_timestamp() AS ingestion_timestamp 
 
        FROM bronze_asset_metadata_latest
    """) 
)




StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9d783416-de2e-4a2e-8125-a0d46772bc2f)

In [15]:
display(spark.sql("""
select * from dbo.silver_telemetry
"""))

display(spark.sql("""
select * from dbo.silver_events
"""))

display(spark.sql("""
select * from dbo.silver_asset_metadata

"""))

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 033cce8d-53a5-4018-9edf-458c3e72b309)

SynapseWidget(Synapse.DataFrame, 5e31dade-0e81-4039-b832-25b28fea9d3e)

SynapseWidget(Synapse.DataFrame, 4174f697-3190-4c9e-bd3f-57e7cd9473f3)

In [16]:
display(spark.sql("""
DESCRIBE dbo.silver_telemetry

"""))
display(spark.sql("""
DESCRIBE  dbo.silver_events

"""))
display(spark.sql("""
DESCRIBE dbo.silver_asset_metadata

"""))

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9b566c6-77d4-45e1-a7a4-886955ca1768)

SynapseWidget(Synapse.DataFrame, 51d33ba4-1dfb-4612-b7e4-c2f92e52af62)

SynapseWidget(Synapse.DataFrame, deb351d7-241f-4656-ab65-153d7987b24a)

In [17]:
from pyspark.sql import functions as F

# ============================================================
# INITIALIZE SILVER WATERMARK
# ============================================================
# The Silver layer already contains data from the initial load.
# Therefore, we initialize the watermark using the maximum
# ingestion_timestamp currently present in each Silver table.
#
# This ensures that existing records are NOT processed again
# during the first incremental execution.
# ============================================================

watermark_df = spark.sql("""
    SELECT
        'telemetry' AS table_name,
        MAX(ingestion_timestamp) AS last_ingested_at
    FROM silver_telemetry

    UNION ALL

    SELECT
        'events' AS table_name,
        MAX(ingestion_timestamp) AS last_ingested_at
    FROM silver_events

    UNION ALL

    SELECT
        'asset_metadata' AS table_name,
        MAX(ingestion_timestamp) AS last_ingested_at
    FROM silver_asset_metadata
""")

# ============================================================
# SAVE THE WATERMARK CONTROL TABLE
# ============================================================

watermark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_watermark")

display(watermark_df)

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 974f1b76-ab17-40a7-8b1a-c75a3b742a29)

In [18]:
from pyspark.sql import functions as F

TABLE_NAME = "telemetry"

last_watermark = spark.sql(f"""
SELECT last_ingested_at
FROM silver_watermark
WHERE table_name = '{TABLE_NAME}'
""").first()["last_ingested_at"]

print(last_watermark)

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 20, Finished, Available, Finished, False)

2026-08-20 08:02:55.687949


In [19]:
telemetry_df = telemetry_df.filter(
    F.col("_ingested_at") > F.lit(last_watermark)
)
display(telemetry_df)

print("Total Records:",
      telemetry_df.count())


StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 219116b5-50a5-4924-aaa8-0d6ac06cea27)

Total Records: 0


In [20]:
silver_incremental_df = telemetry_df.select(
    F.col("timestamp").alias("raw_timestamp"),

    F.coalesce(
        F.to_timestamp("timestamp","dd-MM-yyyy HH:mm"),
        F.to_timestamp("timestamp","dd-MM-yyyy HH:mm:ss"),
        F.to_timestamp("timestamp","yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("timestamp","yyyy-MM-dd HH:mm:ss.SSS")
    ).alias("timestamp"),

    "site_id",
    "building_id",
    "asset_id",
    "sensor_id",

    F.col("temperature").cast("double").alias("temperature"),
    F.col("humidity").cast("double").alias("humidity"),
    F.col("pressure").cast("double").alias("pressure"),
    F.col("vibration").cast("double").alias("vibration"),
    F.col("power_consumption").cast("double").alias("power_consumption"),

    F.col("operating_mode"),

    F.col("_source_file").alias("source_file"),
    F.col("_ingested_at").alias("ingestion_timestamp")
)

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 22, Finished, Available, Finished, False)

In [21]:
if silver_incremental_df.count() > 0:

    silver_incremental_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("silver_telemetry")

    print("Incremental load completed")

else:
    print("No new records found")

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 23, Finished, Available, Finished, False)

No new records found


In [22]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Get latest processed timestamp
new_watermark = spark.table("dbo.silver_telemetry").select(
    F.max("ingestion_timestamp")
).first()[0]

print("New Watermark:", new_watermark)

# Update watermark table
watermark_tbl = DeltaTable.forName(
    spark,
    "silver_watermark"
)

watermark_tbl.update(
    condition="table_name = 'telemetry'",
    set={
        "last_ingested_at": F.lit(new_watermark)
    }
)

# Verify
display(spark.table("silver_watermark"))


StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 24, Finished, Available, Finished, False)

New Watermark: 2026-08-20 08:02:55.687949


SynapseWidget(Synapse.DataFrame, fad9851f-d82f-49c6-8a74-54a6149591a2)

In [23]:
before_count = spark.table("dbo.silver_telemetry").count()

print("Before Count:", before_count)



after_count = spark.table("dbo.silver_telemetry").count()

print("After Count:", after_count)

print("Records Added:", after_count - before_count)

StatementMeta(, b6083940-b71a-4220-89ff-afbda284c427, 25, Finished, Available, Finished, False)

Before Count: 6000
After Count: 6000
Records Added: 0
